In [ ]:
#Python v3.10.11
#Import necessary packages

import os
import sys
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import matplotlib.image as mpimg
import matplotlib as mpl

sys.path.append('functions')

from death_prediction_functions import time_to_death_grouped, cross_validation, train_nn, generate_nn_pred
from gene_analysis_functions import get_great, get_cistrome, get_pos, insig_nan

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
#set directory and get relevant files

os.chdir('/Users/samanderson/Desktop/pellegrini_lab_research/model_outputs')
death_pvals = pd.read_excel('death_classifier_probes.xlsx', index_col=0)
pinv_pvals = pd.read_excel('pseudoinverse_probes_filtered.xlsx', index_col=0)

In [ ]:
#blood glucose probes
GLU_probes = pinv_pvals['M10_poststress_GLU_pval']
GLU_probes = GLU_probes.dropna()
GLU_probes = list(GLU_probes.index)

#aging probes
age_probes = death_pvals
age_probes = age_probes.dropna()
print(age_probes)

age_probes = list(age_probes.index)

intersections = list(set(GLU_probes) & set(age_probes))
cross_identified_probes = death_pvals.loc[intersections]
cross_identified_probes = cross_identified_probes[['coef', 'associated_genes']]
cross_identified_probes

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import re

# Extract genes from associated_genes column
def extract_genes(gene_string):
    # Use regex to match gene names with or without coordinates
    gene_matches = re.findall(r'([A-Za-z0-9]+)\s*\(\+\d+,\d+\)?', gene_string)
    genes = [match for match in gene_matches]  # Extract gene names
    return genes

# Apply the function to each entry in 'associated_genes' column
associated_genes_full = [eval(gene) for gene in cross_identified_probes['associated_genes']]
cross_identified_probes['genes'] = cross_identified_probes['associated_genes'].apply(extract_genes)

# Count occurrences of each gene across all entries
gene_counts = pd.Series([gene for genes_list in cross_identified_probes['genes'] for gene in genes_list]).value_counts()

# Determine duplication status of each gene
is_duplicated = {gene: count > 1 for gene, count in gene_counts.items()}

# Define marker style based on duplication status
marker_style = ['o' if any(is_duplicated[gene] for gene in genes) else '^' for genes in cross_identified_probes['genes']]

# Create the scatterplot with emphasized duplicated genes
fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=cross_identified_probes, x='coef', y='ID', hue='coef', palette='viridis_d', 
                style=marker_style, s=200, legend=False)

# Annotate each point with associated genes
for i, genes in enumerate(cross_identified_probes['genes']):
    x = cross_identified_probes['coef'][i]
    y = cross_identified_probes.index
    annot = ', '.join(genes)  # Join the list of g

for i, genes in enumerate(associated_genes_full):
    x = cross_identified_probes['coef'][i]
    y = cross_identified_probes.index[i]
    annot = ', '.join(genes)  # Join the list of genes into a string
    ax.annotate(annot, (x, y), textcoords="offset points", xytext=(0, 10), ha='left')

plt.xlabel('coef', size=15)
plt.ylabel('probes', size=15)

ax.set_facecolor((1, 1, 0.8509803921))

plt.show()